# Generative Models: VAEs and GANs

**Module 13: Generative Models**  
**Objective**: Understanding generative modeling from scratch

Generative Models:
- Variational Autoencoders (VAEs)
- Generative Adversarial Networks (GANs)
- Mode collapse and training stability
- Latent space manipulation
- Conditional generation

## What You'll Learn
1. VAE architecture and ELBO loss
2. Reparameterization trick
3. GAN architecture (generator, discriminator)
4. GAN training dynamics
5. Common GAN variants (DCGAN, WGAN, StyleGAN)
6. Evaluation metrics (FID, IS)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)
np.random.seed(42)

## 1. Variational Autoencoders (VAEs)

**VAE** learns a probabilistic mapping between data and latent space.

**Architecture**:
```
Encoder: x → μ, σ (mean and std of latent distribution)
Sampling: z ~ N(μ, σ) via reparameterization
Decoder: z → x̂ (reconstruct input)
```

**Loss (ELBO)**:

$$
\mathcal{L} = \underbrace{\|x - x̂\|^2}_{\text{Reconstruction}} + \underbrace{\text{KL}(q(z|x) \| p(z))}_{\text{Regularization}}
$$

Where:
- Reconstruction: How well we reconstruct input
- KL divergence: How close latent distribution is to prior N(0,1)

In [ ]:
class VAE(nn.Module):
    """
    Variational Autoencoder.
    
    Learns probabilistic latent representation of data.
    """
    
    def __init__(self, input_dim: int = 784, hidden_dim: int = 400, latent_dim: int = 20):
        super().__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Latent distribution parameters
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()  # Output in [0, 1]
        )
        
    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Encode input to latent distribution parameters.
        
        Args:
            x: (batch_size, input_dim)
            
        Returns:
            mu: (batch_size, latent_dim)
            logvar: (batch_size, latent_dim)
        """
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """
        Reparameterization trick: z = μ + σ * ε, where ε ~ N(0,1)
        
        This makes sampling differentiable.
        
        Args:
            mu: (batch_size, latent_dim)
            logvar: (batch_size, latent_dim)
            
        Returns:
            z: (batch_size, latent_dim)
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z
    
    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """
        Decode latent variable to reconstruction.
        
        Args:
            z: (batch_size, latent_dim)
            
        Returns:
            x_recon: (batch_size, input_dim)
        """
        return self.decoder(z)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Forward pass through VAE.
        
        Args:
            x: (batch_size, input_dim)
            
        Returns:
            x_recon: (batch_size, input_dim)
            mu: (batch_size, latent_dim)
            logvar: (batch_size, latent_dim)
        """
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar
    
    def loss_function(self, x: torch.Tensor, x_recon: torch.Tensor, 
                     mu: torch.Tensor, logvar: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Compute VAE loss (ELBO).
        
        Loss = Reconstruction + KL Divergence
        
        Args:
            x: Original input
            x_recon: Reconstructed input
            mu: Latent mean
            logvar: Latent log variance
            
        Returns:
            loss: Total loss
            recon_loss: Reconstruction loss
            kl_loss: KL divergence
        """
        # Reconstruction loss (Binary Cross Entropy)
        recon_loss = F.binary_cross_entropy(x_recon, x, reduction='sum')
        
        # KL divergence: KL(q(z|x) || p(z))
        # = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        
        loss = recon_loss + kl_loss
        
        return loss, recon_loss, kl_loss
    
    def sample(self, num_samples: int = 16) -> torch.Tensor:
        """
        Generate new samples from prior.
        
        Args:
            num_samples: Number of samples to generate
            
        Returns:
            samples: (num_samples, input_dim)
        """
        with torch.no_grad():
            # Sample from prior N(0, I)
            z = torch.randn(num_samples, self.fc_mu.out_features).to(device)
            samples = self.decode(z)
        return samples

# Create VAE
vae = VAE(input_dim=784, hidden_dim=400, latent_dim=20).to(device)

total_params = sum(p.numel() for p in vae.parameters())
print("VAE Architecture:")
print("="*70)
print(f"Total parameters: {total_params:,}")
print(f"Latent dimension: 20")
print(f"\n{vae}")

# Test forward pass
batch_size = 32
test_input = torch.rand(batch_size, 784).to(device)

x_recon, mu, logvar = vae(test_input)
loss, recon_loss, kl_loss = vae.loss_function(test_input, x_recon, mu, logvar)

print(f"\nForward pass:")
print(f"  Input: {test_input.shape}")
print(f"  Reconstruction: {x_recon.shape}")
print(f"  Latent mean (μ): {mu.shape}")
print(f"  Latent logvar (log σ²): {logvar.shape}")
print(f"\nLoss components:")
print(f"  Total loss: {loss.item():.2f}")
print(f"  Reconstruction loss: {recon_loss.item():.2f}")
print(f"  KL divergence: {kl_loss.item():.2f}")

## 2. Reparameterization Trick

**Problem**: Can't backprop through random sampling z ~ N(μ, σ)

**Solution**: Reparameterize as z = μ + σ · ε, where ε ~ N(0,1)

Now gradient can flow through μ and σ!

In [ ]:
def visualize_reparameterization():
    """Visualize reparameterization trick."""
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # 1. Problem: Non-differentiable sampling
    ax1 = axes[0]
    ax1.axis('off')
    ax1.set_title('Problem: Direct Sampling', fontsize=12, weight='bold')
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    
    # Draw boxes
    boxes = [
        (0.1, 0.7, 0.25, 0.15, "Encoder\nx → μ, σ", 'lightblue'),
        (0.4, 0.7, 0.2, 0.15, "Sample\nz ~ N(μ, σ)", 'lightcoral'),
        (0.7, 0.7, 0.2, 0.15, "Decoder\nz → x̂", 'lightgreen')
    ]
    
    for x, y, w, h, label, color in boxes:
        rect = plt.Rectangle((x, y), w, h, color=color, ec='black', linewidth=2)
        ax1.add_patch(rect)
        lines = label.split('\n')
        for i, line in enumerate(lines):
            ax1.text(x + w/2, y + h/2 + (0.03 if len(lines)==2 else 0) - i*0.05, 
                    line, ha='center', va='center', fontsize=9, weight='bold')
    
    # Draw arrows
    ax1.annotate('', xy=(0.4, 0.775), xytext=(0.37, 0.775),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    ax1.annotate('', xy=(0.7, 0.775), xytext=(0.62, 0.775),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    
    # Red X on sampling
    ax1.plot([0.45, 0.55], [0.72, 0.83], 'r-', linewidth=4)
    ax1.plot([0.45, 0.55], [0.83, 0.72], 'r-', linewidth=4)
    ax1.text(0.5, 0.6, '✗ No gradient!', ha='center', fontsize=11, 
            color='red', weight='bold')
    
    # 2. Solution: Reparameterization
    ax2 = axes[1]
    ax2.axis('off')
    ax2.set_title('Solution: Reparameterization', fontsize=12, weight='bold')
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    
    boxes2 = [
        (0.1, 0.7, 0.25, 0.15, "Encoder\nx → μ, σ", 'lightblue'),
        (0.4, 0.8, 0.15, 0.1, "ε ~ N(0,1)", 'lightyellow'),
        (0.4, 0.6, 0.2, 0.15, "z = μ + σ·ε", 'lightgreen'),
        (0.7, 0.7, 0.2, 0.15, "Decoder\nz → x̂", 'lightcoral')
    ]
    
    for x, y, w, h, label, color in boxes2:
        rect = plt.Rectangle((x, y), w, h, color=color, ec='black', linewidth=2)
        ax2.add_patch(rect)
        lines = label.split('\n')
        for i, line in enumerate(lines):
            ax2.text(x + w/2, y + h/2 + (0.02 if len(lines)==2 else 0) - i*0.04, 
                    line, ha='center', va='center', fontsize=9, weight='bold')
    
    # Draw arrows
    ax2.annotate('', xy=(0.4, 0.73), xytext=(0.37, 0.73),
                arrowprops=dict(arrowstyle='->', lw=2, color='blue'))
    ax2.annotate('', xy=(0.475, 0.73), xytext=(0.475, 0.77),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    ax2.annotate('', xy=(0.7, 0.73), xytext=(0.62, 0.73),
                arrowprops=dict(arrowstyle='->', lw=2, color='blue'))
    
    # Green check
    ax2.text(0.5, 0.5, '✓ Differentiable!', ha='center', fontsize=11, 
            color='green', weight='bold')
    
    # 3. Gradient flow
    ax3 = axes[2]
    ax3.set_title('Gradient Flow', fontsize=12, weight='bold')
    ax3.set_xlabel('Parameter Value', fontsize=10)
    ax3.set_ylabel('Loss', fontsize=10)
    
    # Plot loss surface
    x = np.linspace(-2, 2, 100)
    loss = x**2 + 1
    ax3.plot(x, loss, 'b-', linewidth=2, label='Loss landscape')
    
    # Show gradient
    point_x = 1.0
    point_y = point_x**2 + 1
    ax3.plot(point_x, point_y, 'ro', markersize=10, label='Current point')
    
    # Gradient arrow
    gradient = 2 * point_x
    ax3.arrow(point_x, point_y, -0.3, -0.3*gradient, head_width=0.1, 
             head_length=0.2, fc='red', ec='red', linewidth=2)
    ax3.text(point_x - 0.5, point_y if gradient > 0 else point_y - 0.5, 
            'Gradient', fontsize=10, color='red', weight='bold')
    
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    
    plt.tight_layout()
    plt.show()
    
    print("\nReparameterization Trick:")
    print("="*70)
    print("\nDirect sampling: z ~ N(μ, σ)")
    print("  ✗ Sampling is stochastic operation")
    print("  ✗ No gradient can flow through random sampling")
    print("  ✗ Can't train with backpropagation")
    
    print("\nReparameterization: z = μ + σ · ε, where ε ~ N(0,1)")
    print("  ✓ Stochasticity moved to ε (independent of parameters)")
    print("  ✓ Gradients flow through μ and σ")
    print("  ✓ Can train end-to-end with backprop")
    
    print("\nMathematically equivalent:")
    print("  Both sample from same distribution N(μ, σ²)")
    print("  But reparameterized version is differentiable!")

visualize_reparameterization()

# Demonstrate gradient flow
def test_gradient_flow():
    """Test gradient flow with and without reparameterization."""
    
    print("\n" + "="*70)
    print("GRADIENT FLOW TEST")
    print("="*70)
    
    # Parameters
    mu = torch.tensor([0.0], requires_grad=True)
    logvar = torch.tensor([0.0], requires_grad=True)
    
    # Sample with reparameterization
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    z = mu + eps * std
    
    # Dummy loss
    loss = z.sum()
    loss.backward()
    
    print("\nWith Reparameterization:")
    print(f"  μ gradient: {mu.grad.item():.4f}")
    print(f"  log(σ²) gradient: {logvar.grad.item():.4f}")
    print("  ✓ Gradients successfully computed!")
    
    print("\nWithout reparameterization:")
    print("  Would get: RuntimeError - can't backprop through sampling")
    print("  ✗ Training impossible!")

test_gradient_flow()

## 3. Generative Adversarial Networks (GANs)

**GAN** = Two networks in competition

**Architecture**:
- **Generator (G)**: Creates fake samples from noise
- **Discriminator (D)**: Classifies real vs fake

**Training**: Minimax game

$$
\min_G \max_D \mathbb{E}_{x\sim p_{data}}[\log D(x)] + \mathbb{E}_{z\sim p_z}[\log(1 - D(G(z)))]
$$

**Intuition**:
- D tries to maximize: Classify real as real, fake as fake
- G tries to minimize: Fool D into thinking fakes are real

In [ ]:
class Generator(nn.Module):
    """
    GAN Generator: noise → fake images.
    """
    
    def __init__(self, latent_dim: int = 100, hidden_dim: int = 256, output_dim: int = 784):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim * 2),
            
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim * 2),
            
            nn.Linear(hidden_dim * 2, output_dim),
            nn.Tanh()  # Output in [-1, 1]
        )
    
    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """
        Args:
            z: (batch_size, latent_dim) noise vectors
            
        Returns:
            fake_images: (batch_size, output_dim)
        """
        return self.model(z)

class Discriminator(nn.Module):
    """
    GAN Discriminator: images → real/fake classification.
    """
    
    def __init__(self, input_dim: int = 784, hidden_dim: int = 256):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()  # Output probability
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch_size, input_dim) images
            
        Returns:
            prob: (batch_size, 1) probability of being real
        """
        return self.model(x)

class GAN:
    """
    Generative Adversarial Network.
    """
    
    def __init__(self, latent_dim: int = 100, data_dim: int = 784):
        self.latent_dim = latent_dim
        
        self.generator = Generator(latent_dim, hidden_dim=256, output_dim=data_dim).to(device)
        self.discriminator = Discriminator(input_dim=data_dim, hidden_dim=256).to(device)
        
        # Separate optimizers
        self.g_optimizer = optim.Adam(self.generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
        self.d_optimizer = optim.Adam(self.discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))
        
        self.criterion = nn.BCELoss()
        
    def train_step(self, real_images: torch.Tensor) -> Tuple[float, float]:
        """
        One training step for GAN.
        
        Args:
            real_images: (batch_size, data_dim)
            
        Returns:
            d_loss: Discriminator loss
            g_loss: Generator loss
        """
        batch_size = real_images.size(0)
        
        # Labels
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)
        
        # ---------------------
        #  Train Discriminator
        # ---------------------
        self.d_optimizer.zero_grad()
        
        # Real images
        real_output = self.discriminator(real_images)
        d_loss_real = self.criterion(real_output, real_labels)
        
        # Fake images
        z = torch.randn(batch_size, self.latent_dim).to(device)
        fake_images = self.generator(z).detach()  # Detach to avoid training G
        fake_output = self.discriminator(fake_images)
        d_loss_fake = self.criterion(fake_output, fake_labels)
        
        # Total discriminator loss
        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        self.d_optimizer.step()
        
        # -----------------
        #  Train Generator
        # -----------------
        self.g_optimizer.zero_grad()
        
        # Generate fake images
        z = torch.randn(batch_size, self.latent_dim).to(device)
        fake_images = self.generator(z)
        
        # Try to fool discriminator
        fake_output = self.discriminator(fake_images)
        g_loss = self.criterion(fake_output, real_labels)  # Want D to think they're real
        
        g_loss.backward()
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item()
    
    def generate(self, num_samples: int = 16) -> torch.Tensor:
        """Generate fake samples."""
        self.generator.eval()
        with torch.no_grad():
            z = torch.randn(num_samples, self.latent_dim).to(device)
            samples = self.generator(z)
        self.generator.train()
        return samples

# Create GAN
gan = GAN(latent_dim=100, data_dim=784)

g_params = sum(p.numel() for p in gan.generator.parameters())
d_params = sum(p.numel() for p in gan.discriminator.parameters())

print("GAN Architecture:")
print("="*70)
print(f"Generator parameters: {g_params:,}")
print(f"Discriminator parameters: {d_params:,}")
print(f"Total parameters: {g_params + d_params:,}")

print("\nGenerator:")
print(gan.generator)

print("\nDiscriminator:")
print(gan.discriminator)

# Test training step
test_real = torch.randn(32, 784).to(device)
d_loss, g_loss = gan.train_step(test_real)

print(f"\nTest training step:")
print(f"  Discriminator loss: {d_loss:.4f}")
print(f"  Generator loss: {g_loss:.4f}")

## 4. GAN Training Dynamics

Training GANs is notoriously unstable. Common issues:
- **Mode collapse**: G produces limited variety
- **Vanishing gradients**: D too strong, G can't learn
- **Oscillation**: Losses don't converge

In [ ]:
def visualize_gan_training():
    """Visualize GAN training dynamics."""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # 1. Training dynamics
    ax1 = axes[0, 0]
    ax1.set_title('GAN Training Losses', fontsize=12, weight='bold')
    ax1.set_xlabel('Training Steps', fontsize=10)
    ax1.set_ylabel('Loss', fontsize=10)
    
    # Simulate training curves
    np.random.seed(42)
    steps = np.arange(0, 100)
    d_loss = 0.7 + 0.3 * np.sin(steps / 10) + np.random.randn(100) * 0.1
    g_loss = 1.0 + 0.4 * np.cos(steps / 10) + np.random.randn(100) * 0.1
    
    d_loss = np.clip(d_loss, 0.2, 1.5)
    g_loss = np.clip(g_loss, 0.2, 2.0)
    
    ax1.plot(steps, d_loss, 'b-', linewidth=2, label='Discriminator', alpha=0.8)
    ax1.plot(steps, g_loss, 'r-', linewidth=2, label='Generator', alpha=0.8)
    ax1.axhline(y=0.693, color='gray', linestyle='--', linewidth=1, 
               label='Optimal (log(2))')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 2.5)
    
    # 2. Mode collapse
    ax2 = axes[0, 1]
    ax2.set_title('Mode Collapse', fontsize=12, weight='bold')
    ax2.set_xlabel('Feature 1', fontsize=10)
    ax2.set_ylabel('Feature 2', fontsize=10)
    
    # Real data (multiple modes)
    modes = [(0, 0), (3, 3), (-3, 3), (0, -3)]
    for mx, my in modes:
        samples = np.random.randn(50, 2) * 0.5 + np.array([mx, my])
        ax2.scatter(samples[:, 0], samples[:, 1], alpha=0.5, s=30, label='') 
    
    # Generated data (collapsed to one mode)
    collapsed = np.random.randn(200, 2) * 0.3
    ax2.scatter(collapsed[:, 0], collapsed[:, 1], color='red', alpha=0.6, 
               s=20, marker='x', label='Generated (collapsed)')
    
    ax2.scatter([], [], color='blue', alpha=0.5, s=30, label='Real data (4 modes)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(-5, 5)
    ax2.set_ylim(-5, 5)
    
    # 3. Discriminator strength
    ax3 = axes[1, 0]
    ax3.set_title('Discriminator Strength Over Training', fontsize=12, weight='bold')
    ax3.set_xlabel('Training Steps', fontsize=10)
    ax3.set_ylabel('Accuracy', fontsize=10)
    
    steps = np.arange(0, 100)
    # D starts strong, then balances
    d_acc = 0.95 - 0.3 * (1 - np.exp(-steps / 20)) + np.random.randn(100) * 0.02
    d_acc = np.clip(d_acc, 0.5, 1.0)
    
    ax3.plot(steps, d_acc, 'b-', linewidth=2, label='D accuracy')
    ax3.axhline(y=0.5, color='green', linestyle='--', linewidth=2, 
               label='Balanced (50%)')
    ax3.fill_between(steps, 0.45, 0.55, alpha=0.2, color='green', 
                     label='Healthy range')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0.4, 1.0)
    
    # 4. Generation quality
    ax4 = axes[1, 1]
    ax4.axis('off')
    ax4.set_title('Generation Quality Over Time', fontsize=12, weight='bold')
    
    stages = [
        (0.15, "Early: Noise", "Generator outputs\nrandom noise"),
        (0.45, "Mid: Shapes", "Basic shapes\nemerging"),
        (0.75, "Late: Realistic", "High-quality\nsamples")
    ]
    
    for x, title, desc in stages:
        # Draw box
        rect = plt.Rectangle((x-0.08, 0.6), 0.16, 0.25, 
                            color='lightblue', ec='black', linewidth=2)
        ax4.add_patch(rect)
        ax4.text(x, 0.9, title, ha='center', va='center', 
                fontsize=10, weight='bold')
        ax4.text(x, 0.7, desc, ha='center', va='center', fontsize=8)
        
        # Draw progress arrow
        if x < 0.7:
            ax4.annotate('', xy=(x+0.12, 0.725), xytext=(x+0.09, 0.725),
                        arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.show()
    
    print("\nGAN Training Challenges:")
    print("="*70)
    
    print("\n1. MODE COLLAPSE")
    print("  • Generator learns to produce limited variety")
    print("  • All samples look similar")
    print("  • Solution: Minibatch discrimination, unrolled GAN")
    
    print("\n2. VANISHING GRADIENTS")
    print("  • Discriminator too strong → G gets no learning signal")
    print("  • G loss saturates")
    print("  • Solution: Wasserstain GAN (WGAN), non-saturating loss")
    
    print("\n3. OSCILLATION")
    print("  • Losses don't converge, keep oscillating")
    print("  • Difficult to determine when to stop")
    print("  • Solution: Careful hyperparameter tuning, spectral norm")
    
    print("\n4. TRAINING INSTABILITY")
    print("  • Sensitive to hyperparameters")
    print("  • May suddenly collapse during training")
    print("  • Solution: Progressive growing, regularization")

visualize_gan_training()

## 5. GAN Variants

**DCGAN** (Deep Convolutional GAN):
- Use Conv layers instead of fully-connected
- BatchNorm in G, no BN in D
- LeakyReLU in D, ReLU in G
- No pooling layers (use strided convolutions)

**WGAN** (Wasserstein GAN):
- Use Wasserstein distance instead of JS divergence
- Clip discriminator weights or use gradient penalty
- More stable training

**StyleGAN**:
- Style-based generator
- Progressive growing
- Excellent high-resolution images

In [ ]:
class DCGANGenerator(nn.Module):
    """
    DCGAN Generator with convolutional layers.
    """
    
    def __init__(self, latent_dim: int = 100, channels: int = 1):
        super().__init__()
        
        self.init_size = 7  # Initial spatial size
        self.latent_dim = latent_dim
        
        self.fc = nn.Linear(latent_dim, 128 * self.init_size ** 2)
        
        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(128),
            
            # Upsample to 14x14
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            # Upsample to 28x28
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            # Output layer
            nn.Conv2d(64, channels, 3, stride=1, padding=1),
            nn.Tanh()
        )
    
    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """
        Args:
            z: (batch_size, latent_dim)
            
        Returns:
            images: (batch_size, channels, 28, 28)
        """
        out = self.fc(z)
        out = out.view(out.size(0), 128, self.init_size, self.init_size)
        img = self.conv_blocks(out)
        return img

class DCGANDiscriminator(nn.Module):
    """
    DCGAN Discriminator with convolutional layers.
    """
    
    def __init__(self, channels: int = 1):
        super().__init__()
        
        self.model = nn.Sequential(
            # 28x28 → 14x14
            nn.Conv2d(channels, 64, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            # 14x14 → 7x7
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            # 7x7 → 1x1
            nn.Conv2d(128, 1, 7, stride=1, padding=0),
            nn.Sigmoid()
        )
    
    def forward(self, img: torch.Tensor) -> torch.Tensor:
        """
        Args:
            img: (batch_size, channels, 28, 28)
            
        Returns:
            validity: (batch_size, 1, 1, 1)
        """
        validity = self.model(img)
        return validity.view(validity.size(0), -1)

# Create DCGAN
dcgan_g = DCGANGenerator(latent_dim=100, channels=1).to(device)
dcgan_d = DCGANDiscriminator(channels=1).to(device)

g_params = sum(p.numel() for p in dcgan_g.parameters())
d_params = sum(p.numel() for p in dcgan_d.parameters())

print("DCGAN Architecture:")
print("="*70)
print(f"Generator parameters: {g_params:,}")
print(f"Discriminator parameters: {d_params:,}")

# Test
test_z = torch.randn(16, 100).to(device)
with torch.no_grad():
    fake_images = dcgan_g(test_z)
    fake_output = dcgan_d(fake_images)

print(f"\nForward pass:")
print(f"  Noise: {test_z.shape}")
print(f"  Generated images: {fake_images.shape}")
print(f"  Discriminator output: {fake_output.shape}")

# Comparison table
print("\n" + "="*70)
print("GAN VARIANTS COMPARISON")
print("="*70)

variants = [
    ("Vanilla GAN", "Fully-connected", "Binary CE", "Unstable", "MNIST"),
    ("DCGAN", "Convolutional", "Binary CE", "More stable", "Images"),
    ("WGAN", "Convolutional", "Wasserstein", "Stable", "Images"),
    ("WGAN-GP", "Convolutional", "Wasserstein + GP", "Very stable", "Images"),
    ("StyleGAN", "Style-based", "Non-saturating", "Stable", "High-res"),
    ("Progressive GAN", "Progressive", "Non-saturating", "Stable", "High-res")
]

print(f"\n{'Variant':<20} {'Architecture':<18} {'Loss':<20} {'Stability':<15} {'Best For':<15}")
print("-" * 90)
for name, arch, loss, stability, best_for in variants:
    print(f"{name:<20} {arch:<18} {loss:<20} {stability:<15} {best_for:<15}")

## 6. Evaluation Metrics

**FID (Fréchet Inception Distance)**:
- Measures distance between real and generated distributions in Inception feature space
- Lower is better

**IS (Inception Score)**:
- Measures quality and diversity
- Higher is better

**Precision & Recall**:
- Precision: Generated samples within real distribution
- Recall: Coverage of real distribution

In [ ]:
def compute_fid_score(real_features: torch.Tensor, fake_features: torch.Tensor) -> float:
    """
    Compute Fréchet Inception Distance (simplified).
    
    FID = ||μ_r - μ_f||² + Tr(Σ_r + Σ_f - 2(Σ_r Σ_f)^(1/2))
    
    Args:
        real_features: (N, feature_dim)
        fake_features: (M, feature_dim)
        
    Returns:
        fid: FID score
    """
    # Compute mean and covariance
    mu_real = real_features.mean(dim=0)
    mu_fake = fake_features.mean(dim=0)
    
    sigma_real = torch.cov(real_features.T)
    sigma_fake = torch.cov(fake_features.T)
    
    # Compute FID (simplified - full version uses matrix square root)
    diff = mu_real - mu_fake
    mean_dist = torch.sum(diff ** 2)
    
    # Trace of covariance difference (simplified)
    cov_dist = torch.trace(sigma_real + sigma_fake)
    
    fid = mean_dist + cov_dist
    
    return fid.item()

# Simulate features
real_features = torch.randn(1000, 2048)
fake_features_good = real_features + torch.randn(1000, 2048) * 0.1
fake_features_bad = torch.randn(1000, 2048) * 2

fid_good = compute_fid_score(real_features, fake_features_good)
fid_bad = compute_fid_score(real_features, fake_features_bad)

print("GAN Evaluation Metrics:")
print("="*70)

print("\nFréchet Inception Distance (FID):")
print(f"  Good generator: {fid_good:.2f} (lower is better)")
print(f"  Bad generator: {fid_bad:.2f}")
print(f"  Improvement: {(fid_bad - fid_good) / fid_bad * 100:.1f}%")

# Visualize FID concept
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Good generator
ax1 = axes[0]
ax1.set_title('Good Generator (Low FID)', fontsize=12, weight='bold')
ax1.scatter(real_features[:100, 0], real_features[:100, 1], 
           alpha=0.5, label='Real', s=30)
ax1.scatter(fake_features_good[:100, 0], fake_features_good[:100, 1], 
           alpha=0.5, label='Generated', s=30, marker='x')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')

# Bad generator
ax2 = axes[1]
ax2.set_title('Bad Generator (High FID)', fontsize=12, weight='bold')
ax2.scatter(real_features[:100, 0], real_features[:100, 1], 
           alpha=0.5, label='Real', s=30)
ax2.scatter(fake_features_bad[:100, 0], fake_features_bad[:100, 1], 
           alpha=0.5, label='Generated', s=30, marker='x')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print("\nOther Metrics:")
print("-"*70)
print("\nInception Score (IS):")
print("  • Measures quality and diversity")
print("  • IS = exp(E[KL(p(y|x) || p(y))])")
print("  • Higher is better (>5 is good)")

print("\nPrecision & Recall:")
print("  • Precision: Are generated samples realistic?")
print("  • Recall: Do we cover the full distribution?")
print("  • Both should be high (~0.8-0.9)")

print("\nHuman Evaluation:")
print("  • Ultimate metric: Do humans think samples are real?")
print("  • Used in competitions and research papers")

## Summary

You've mastered VAEs and GANs:
- ✅ VAE architecture (encoder, decoder, latent space)
- ✅ ELBO loss (reconstruction + KL divergence)
- ✅ Reparameterization trick (enables backpropagation)
- ✅ GAN architecture (generator vs discriminator)
- ✅ GAN training dynamics (mode collapse, instability)
- ✅ GAN variants (DCGAN, WGAN, StyleGAN)
- ✅ Evaluation metrics (FID, IS, precision/recall)

**Key Insights**:
1. **VAEs** learn probabilistic latent representations with smooth interpolation
2. **Reparameterization** makes VAE training differentiable
3. **GANs** use adversarial training for high-quality generation
4. **Training GANs** is unstable - mode collapse and vanishing gradients
5. **DCGAN** uses convolutional architecture for image generation
6. **FID score** measures quality of generated distribution

**VAE vs GAN**:
| Aspect | VAE | GAN |
|--------|-----|-----|
| Training | Stable | Unstable |
| Quality | Blurry | Sharp |
| Diversity | High | Risk of mode collapse |
| Latent space | Structured | Less structured |
| Likelihood | Tractable | Intractable |

**Applications**:
- Image generation and editing
- Data augmentation
- Anomaly detection (VAE)
- Super-resolution
- Style transfer
- Drug discovery (molecule generation)

**Next Notebook**: Diffusion models (DDPM, DDIM, Stable Diffusion)

## Exercises

1. **Train VAE on MNIST**: Implement full training loop and visualize latent space  
2. **Conditional GAN**: Extend GAN to generate images of specific classes
3. **Interpolation**: Interpolate between two images in latent space
4. **Fix mode collapse**: Implement unrolled GAN or minibatch discrimination